In [5]:
! ls -al /kaggle/input/notebooks/arjoebasak/keyvalue-ablation1/KeyValue-Extraction

total 28
drwxr-xr-x 11 nobody nogroup    0 Oct 15 00:16  .
drwxr-xr-x  3 nobody nogroup    0 Oct 15 00:16  ..
drwxr-xr-x  2 nobody nogroup    0 Oct 15 00:16  configs
drwxr-xr-x  4 nobody nogroup    0 Oct 15 00:16  data
drwxr-xr-x  7 nobody nogroup    0 Oct 15 00:16  .git
-rw-r--r--  1 nobody nogroup  840 Oct 15 00:16  .gitignore
-rw-r--r--  1 nobody nogroup 5715 Oct 15 00:16 'IE643-2026 Course Project Prep Instructions.txt'
drwxr-xr-x  2 nobody nogroup    0 Oct 15 00:16  notebooks
-rw-r--r--  1 nobody nogroup 2915 Oct 15 00:16 'Project description.txt'
-rw-r--r--  1 nobody nogroup 6528 Oct 15 00:16  README.md
-rw-r--r--  1 nobody nogroup 1062 Oct 15 00:16  requirements.txt
drwxr-xr-x 14 nobody nogroup    0 Oct 15 00:16  results
drwxr-xr-x  2 nobody nogroup    0 Oct 15 00:16  scripts
drwxr-xr-x  3 nobody nogroup    0 Oct 15 00:16  slides
drwxr-xr-x  5 nobody nogroup    0 Oct 15 00:16  src
drwxr-xr-x  3 nobody nogroup    0 Oct 15 00:16  tests


In [1]:
import os, subprocess, sys

REPO_DIR = "/kaggle/working/KeyValue-Extraction"
REPO_URL = "https://github.com/ABasakEE/KeyValue-Extraction.git"

if not os.path.isdir(REPO_DIR):
    print("Cloning latest KeyValue-Extraction with VRDU Ad-Buy fix...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

os.chdir(REPO_DIR)
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

!pip install -q --upgrade pip setuptools wheel
!pip install -q seqeval faker transformers datasets tabulate
print(f"Working directory: {os.getcwd()}")

Cloning latest KeyValue-Extraction with VRDU Ad-Buy fix...


Cloning into '/kaggle/working/KeyValue-Extraction'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 9.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 34.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Working directory: /kaggle/working/KeyValue-Extraction


In [2]:
import json
from pathlib import Path
import pandas as pd

# Location of previous run mounted as dataset
PREV_RESULTS = Path("/kaggle/input/notebooks/arjoebasak/keyvalue-ablation1/KeyValue-Extraction/results")

conditions = [
    ("Previous 12-D Baseline (Ref)", PREV_RESULTS / "xfund_joint_lilt"),
    ("Full 16-D Upgraded Model",      PREV_RESULTS / "xfund_joint_16d"),
    ("w/o Line-of-Sight (14-D)",      PREV_RESULTS / "ablation_no_los"),
    ("w/o Typography (12-D)",         PREV_RESULTS / "ablation_no_typography"),
    ("w/o Containment (11-D)",        PREV_RESULTS / "ablation_no_containment"),
    ("Geometry Only (5-D)",           PREV_RESULTS / "ablation_geometry_only"),
    ("BCE Loss (no Focal)",           PREV_RESULTS / "ablation_bce_loss"),
    ("Independent Decoding (no 1-1)", PREV_RESULTS / "ablation_no_one_to_one"),
    ("FUNSD Joint 16-D",              PREV_RESULTS / "funsd_joint_16d"),
]

rows = []
for label, path in conditions:
    mf = path / "metrics.json"
    if mf.exists():
        d = json.loads(mf.read_text(encoding="utf-8"))
        ent_f1 = d.get("mean_f1", 0.0)
        ent_std = d.get("std_f1", 0.0)
        lnk_f1 = d.get("mean_linking_f1", 0.0)
        lnk_std = d.get("std_linking_f1", 0.0)
        rows.append({
            "Experiment / Condition": label,
            "Entity F1": f"{ent_f1*100:.2f}% ± {ent_std*100:.2f}%",
            "Linking F1": f"{lnk_f1*100:.2f}% ± {lnk_std*100:.2f}%",
            "Status": "Completed"
        })
    else:
        rows.append({
            "Experiment / Condition": label,
            "Entity F1": "--",
            "Linking F1": "--",
            "Status": "Not Found"
        })

df = pd.DataFrame(rows)
print(df.to_markdown(index=False))

# Check VRDU Track B1 Registration Forms Generalization Gap
vrdu_mf = PREV_RESULTS / "vrdu_lilt" / "metrics.json"
if vrdu_mf.exists():
    vdata = json.loads(vrdu_mf.read_text(encoding="utf-8"))
    gap = vdata.get("gap", {})
    print("\n--- Track B1: VRDU Registration Forms Generalization Gap ---")
    print(f"  MTL (Seen):   {gap.get('mtl_mean_f1', 0)*100:.2f}%")
    print(f"  UTL (Unseen): {gap.get('utl_mean_f1', 0)*100:.2f}%")
    print(f"  Gap:          {gap.get('gap_f1', 0)*100:.2f}% ({gap.get('relative_drop_pct', 0):.1f}% drop)")

| Experiment / Condition        | Entity F1      | Linking F1     | Status    |
|:------------------------------|:---------------|:---------------|:----------|
| Previous 12-D Baseline (Ref)  | 79.09% ± 0.21% | 65.51% ± 1.26% | Completed |
| Full 16-D Upgraded Model      | 79.32% ± 0.36% | 67.21% ± 0.83% | Completed |
| w/o Line-of-Sight (14-D)      | 78.60% ± 0.71% | 65.26% ± 2.11% | Completed |
| w/o Typography (12-D)         | 79.09% ± 0.21% | 65.51% ± 1.26% | Completed |
| w/o Containment (11-D)        | 79.25% ± 0.36% | 64.57% ± 1.13% | Completed |
| Geometry Only (5-D)           | 78.87% ± 0.19% | 57.26% ± 1.66% | Completed |
| BCE Loss (no Focal)           | 79.18% ± 0.34% | 64.76% ± 0.64% | Completed |
| Independent Decoding (no 1-1) | 79.32% ± 0.36% | 57.33% ± 0.03% | Completed |
| FUNSD Joint 16-D              | --             | --             | Not Found |

--- Track B1: VRDU Registration Forms Generalization Gap ---
  MTL (Seen):   87.74%
  UTL (Unseen): 66.66%
  Gap:      

In [4]:
import zipfile
from pathlib import Path

# Path to the mounted input results
input_results = Path("/kaggle/input/notebooks/arjoebasak/keyvalue-ablation1/KeyValue-Extraction/results")
out_metrics_zip = Path("/kaggle/working/all_previous_metrics.zip")
out_model_zip = Path("/kaggle/working/best_xfund_16d_model.zip")

# 1. Package ALL metrics JSON and YAML files into a small zip (~2 MB)
json_files = list(input_results.glob("**/*.json")) + list(input_results.glob("**/*.yaml"))
print(f"Found {len(json_files)} metric/config files. Packaging...")

with zipfile.ZipFile(out_metrics_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in json_files:
        arcname = f.relative_to(input_results)
        zf.write(f, arcname)

print(f"[+] Metrics packaged: {out_metrics_zip} ({out_metrics_zip.stat().st_size / 1024:.1f} KB)")

# 2. Package the single best 16-D model checkpoint
candidates = list(input_results.glob("xfund_joint_16d/*/best_model.pt")) + list(input_results.glob("xfund_joint_16d/best_model.pt"))

if candidates:
    best_model_pt = candidates[0]
    print(f"Found checkpoint: {best_model_pt} ({best_model_pt.stat().st_size / 1e6:.1f} MB)")
    with zipfile.ZipFile(out_model_zip, "w", zipfile.ZIP_STORED) as zf:
        zf.write(best_model_pt, "best_model.pt")
    print(f"[+] Best model packaged: {out_model_zip} ({out_model_zip.stat().st_size / 1e6:.1f} MB)")
else:
    print("[-] No best_model.pt found under xfund_joint_16d; only metrics are present.")

Found 148 metric/config files. Packaging...
[+] Metrics packaged: /kaggle/working/all_previous_metrics.zip (101.8 KB)
Found checkpoint: /kaggle/input/notebooks/arjoebasak/keyvalue-ablation1/KeyValue-Extraction/results/xfund_joint_16d/seed_13/best_model.pt (521.7 MB)
[+] Best model packaged: /kaggle/working/best_xfund_16d_model.zip (521.7 MB)


In [5]:
import os, subprocess

vrdu_dir = "data/vrdu"
if not os.path.isdir(vrdu_dir):
    print("Cloning official Google Research VRDU repository...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/google-research-datasets/vrdu.git", vrdu_dir], check=True)

# Quick sanity check on ad-buy-form
from src.data.vrdu_loader import load_vrdu, load_deepform_protocol, summarize
examples, labels, id2label = load_vrdu("data/vrdu", "ad-buy-form")
print("\nVRDU Ad-Buy Summary:", summarize(examples))
protocol = load_deepform_protocol(examples, "data/vrdu/ad-buy-form/few_shot-splits", n_train=200)
print(f"Loaded {len(protocol['MTL'])} MTL folds and {len(protocol['UTL'])} UTL folds successfully!")

Cloning official Google Research VRDU repository...


Cloning into 'data/vrdu'...
Updating files: 100% (2681/2681), done.



VRDU Ad-Buy Summary: {'documents': 641, 'templates': {'ad-buy-form': 641}, 'words': 977153, 'mean_words_per_doc': 1524.4, 'multi_page_docs': 461, 'docs_with_unaligned_entities': 0, 'total_unaligned_entity_occurrences': 0}
Loaded 3 MTL folds and 3 UTL folds successfully!


In [7]:
%%bash
cd /kaggle/working/KeyValue-Extraction
git pull origin main

Updating 859427a..46e0647
Fast-forward
 src/train.py | 239 +++++++++++++++++++++++++++++++++++++++++++++++------------
 1 file changed, 192 insertions(+), 47 deletions(-)


From https://github.com/ABasakEE/KeyValue-Extraction
 * branch            main       -> FETCH_HEAD
   859427a..46e0647  main       -> origin/main


In [10]:
!python src/train.py --config configs/vrdu_adbuy_lilt.yaml --protocol matched

[*] Detected 2 GPU(s): [0, 1] (total visible: 2)
    GPU 0: Tesla T4 (15.6 GB VRAM)
    GPU 1: Tesla T4 (15.6 GB VRAM)
[*] Backbone: lilt | Dataset: vrdu | Task: tagging | Seeds: [13, 21, 42]
VRDU: {"documents": 641, "templates": {"ad-buy-form": 641}, "words": 977153, "mean_words_per_doc": 1524.4, "multi_page_docs": 461, "docs_with_unaligned_entities": 0, "total_unaligned_entity_occurrences": 0}

[Official DeepForm Split] MTL Fold 0 (200 train, 100 val, 300 test)

[Official DeepForm Split] MTL Fold 1 (200 train, 100 val, 300 test)

[Official DeepForm Split] MTL Fold 2 (200 train, 100 val, 300 test)

[Official DeepForm Split] UTL Fold 0 (200 train, 100 val, 159 test)

[Official DeepForm Split] UTL Fold 1 (200 train, 100 val, 157 test)

[Official DeepForm Split] UTL Fold 2 (200 train, 100 val, 153 test)
[*] Resuming from checkpoint: MTL/fold0/seed13 (results/vrdu_adbuy_lilt/MTL_fold0_seed13/result.json) -> Test F1 0.9313
[*] Resuming from checkpoint: MTL/fold0/seed21 (results/vrdu_adbuy_

In [12]:
import json
from pathlib import Path

mf = Path("results/vrdu_adbuy_lilt/metrics.json")
if mf.exists():
    data = json.loads(mf.read_text(encoding="utf-8"))
    gap = data.get("gap", {})
    print("=" * 65)
    print("TRACK B2: VRDU AD-BUY FORMS (DEEPFORM) GENERALIZATION GAP")
    print("=" * 65)
    print(f"MTL (Seen Templates)   : {gap.get('mtl_mean_f1', 0)*100:.2f}% ± {gap.get('mtl_std', 0)*100:.2f}%")
    print(f"UTL (Unseen Templates) : {gap.get('utl_mean_f1', 0)*100:.2f}% ± {gap.get('utl_std', 0)*100:.2f}%")
    print(f"Generalization Gap     : {gap.get('gap_f1', 0)*100:.2f}% ({gap.get('relative_drop_pct', 0):.1f}% relative drop)")
    print("=" * 65)

TRACK B2: VRDU AD-BUY FORMS (DEEPFORM) GENERALIZATION GAP
MTL (Seen Templates)   : 92.46% ± 0.86%
UTL (Unseen Templates) : 90.61% ± 1.12%
Generalization Gap     : 1.85% (2.0% relative drop)


In [13]:
import os
import zipfile
from pathlib import Path

repo_dir = Path("/kaggle/working/KeyValue-Extraction")
out_zip = Path("/kaggle/working/vrdu_adbuy_results.zip")

# Archive all json metrics and configs, skipping heavy .pt model weights to save disk space
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(repo_dir / "results"):
        for file in files:
            if file.endswith((".json", ".yaml", ".txt")):
                full_path = Path(root) / file
                rel_path = full_path.relative_to(repo_dir)
                zf.write(full_path, arcname=str(rel_path))

print(f"[+] Successfully generated archive: {out_zip} ({out_zip.stat().st_size / 1e6:.2f} MB)")

[+] Successfully generated archive: /kaggle/working/vrdu_adbuy_results.zip (0.08 MB)
